# Polymer-detection HITL pipeline

Orchestration notebook. All logic lives in the `polymer_detection` package; this notebook only wires it together. For background read **ARCHITECTURE.md** (the *why*) and **GUIDE.md** (the *how*).

Mode selection happens by editing the `Config` instance in the next cell. Every retrain is from scratch by design (ARCHITECTURE.md §12).

| Step | Use when |
|---|---|
| §1 Setup & sanity | first thing each session |
| §2 Bootstrap mode | cold start / adding a new anchor image |
| §3 Correction round | the main HITL loop: predict → mine → stage |
| §4 Retrain | after you finish painting in ilastik |
| §5 Trajectory + eval | inspect progress, build poster figures |


## §1. Setup & sanity check

In [1]:
import os, sys
PROJECT_ROOT = os.path.dirname(os.path.abspath(''))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from polymer_detection import cfg
# --- mode switches (edit me) -----------------------------------------------
cfg.RUN_MODE          = 'correction'   # 'bootstrap' | 'correction'
cfg.FOLD_MODE         = 'single'       # 'kfold' | 'single' (single carries over last kfold's best fold)
cfg.TARGET_CLASSES    = [0, 3]         # [0] polymer, [3] boundary, [0, 3] both
cfg.MINING_BATCH_SIZE = 32
cfg.PRED_BATCH_SIZE   = 150
cfg.EPOCHS            = 50

# --- val/test split (explicit, polymer-balanced) ---------------------------
# Empty list = legacy "first cfg.TRUTH_VAL_COUNT sorted" rule.
# Non-empty = these exact stems are val, everything else in InitialLabels is
# test. Useful when sorted-first lands on the polymer-light end; here we add
# D16_03_1_17 (polymer-heavy) so val_dice has more signal to grade.
cfg.VAL_STEMS = [
    'D16_03_1_12_Bright Field_001',
    'D16_03_1_13_Bright Field_001',
    'D16_03_1_17_Bright Field_001',
]

# --- model backbone (ceiling-test knob) ------------------------------------
# 'unet'                          — vanilla UNet, 4M params (default)
# 'smp_unet_efficientnet-b0'      — SMP UNet w/ EfficientNet-B0, ~6M params, ImageNet pretrained
# 'smp_unet_efficientnet-b7'      — SMP UNet w/ EfficientNet-B7, ~67M params (drop BATCH_SIZE for VRAM)
# Each backbone trains and saves its own checkpoints (filenames carry the
# backbone tag) — they coexist. Densify still uses cfg.DENSIFY_BACKBONE
# (default 'unet', loop 0) for non-target fills so SMP runs reuse the
# original clean baseline.
cfg.MODEL_BACKBONE    = 'unet'
# cfg.BATCH_SIZE      = 4         # uncomment if b7 OOMs

# --- mini-crops (only matters if you run §3.1) ------------------------------
cfg.MINI_CROP_TARGET_CLASS = 0    # focus class for the fast-label workflow
cfg.MINI_CROP_BATCH_SIZE   = 200  # how many adaptive crops to stage per call

cfg.ensure_dirs()
from sickling.protrusion_detection.splits import build_val_test_pairs
val_pairs, test_pairs = build_val_test_pairs()
print(f'device                              = {cfg.DEVICE}')
print(f'RUN_MODE / FOLD_MODE                = {cfg.RUN_MODE} / {cfg.FOLD_MODE}')
print(f'MODEL_BACKBONE                      = {cfg.MODEL_BACKBONE}')
print(f'TARGET_CLASSES                      = {cfg.TARGET_CLASSES}')
print(f'PRED_BATCH_SIZE / MINING_BATCH_SIZE = {cfg.PRED_BATCH_SIZE} / {cfg.MINING_BATCH_SIZE}')
print(f'MINI_CROP_TARGET_CLASS / BATCH      = {cfg.MINI_CROP_TARGET_CLASS} / {cfg.MINI_CROP_BATCH_SIZE}')
print(f'DENSIFY_BACKBONE/FOLD/LOOP          = {cfg.DENSIFY_BACKBONE} / {cfg.DENSIFY_FOLD} / {cfg.DENSIFY_LOOP}')
print(f'val  ({len(val_pairs)}) : {[os.path.splitext(os.path.basename(p[0]))[0] for p in val_pairs]}')
print(f'test ({len(test_pairs)}) : {[os.path.splitext(os.path.basename(p[0]))[0] for p in test_pairs]}')
print(f'PROMOTE_TILES_TO_VAL                = {cfg.PROMOTE_TILES_TO_VAL}')

device                              = cuda
RUN_MODE / FOLD_MODE                = correction / single
MODEL_BACKBONE                      = unet
TARGET_CLASSES                      = [0, 3]
PRED_BATCH_SIZE / MINING_BATCH_SIZE = 150 / 32
MINI_CROP_TARGET_CLASS / BATCH      = 0 / 200
DENSIFY_BACKBONE/FOLD/LOOP          = unet / 2 / 0
val  (3) : ['D16_03_1_12_Bright Field_001', 'D16_03_1_13_Bright Field_001', 'D16_03_1_17_Bright Field_001']
test (2) : ['D16_03_1_15_Bright Field_001', 'D16_03_1_1_Bright Field_001']
PROMOTE_TILES_TO_VAL                = False


In [2]:
from sickling.protrusion_detection.paths import labeled_stems, list_h5, list_raw
init, boot = labeled_stems()
tiles_todo = list_h5(cfg.TILES_TODO_DIR)
corrected = list_h5(cfg.CORRECTED_TILES_DIR)
pool_raw = list_raw(cfg.CORRECTION_POOL_DIR)
print(f'InitialLabels     = {len(init)}')
print(f'BootstrappedLabels= {len(boot)}')
print(f'CorrectionPool raw= {len(pool_raw)}')
print(f'TilesToBeCorrected= {len(tiles_todo) // 2 if tiles_todo else 0}  (raw + PRED pair)')
print(f'CorrectedTiles    = {len(corrected)}')

InitialLabels     = 5
BootstrappedLabels= 12
CorrectionPool raw= 358
TilesToBeCorrected= 15  (raw + PRED pair)
CorrectedTiles    = 90


## §2. Bootstrap mode (only for cold start / new anchor images)

Trains the throwaway generator on `InitialLabels`, writes starting PREDs for any bootstrap stem that lacks a dense label, then trains the final model from scratch on `BootstrappedLabels`. In the steady state (anchors already painted) phases A & B are no-ops and only the final train runs.

In [ ]:
# §2 — Bootstrap mode. Trains the throwaway generator on InitialLabels
# (optionally + the current pool), writes a starting PRED into
# BootstrappedLabels for new cold-start raws, then (optionally) runs the
# final from-scratch train on the current pool.
#
# In the steady state with all BootstrappedLabels painted, phases A & B are
# a no-op and only Phase C (final train) runs. The final train operates on
# BootstrappedLabels + CorrectedTiles + (densified) mini-crops; InitialLabels
# stays permanently held out as val/test per ARCHITECTURE.md §12.
#
# Options:
#   force_stems        - re-bootstrap a specific raw image (e.g. you deleted
#                        a BootstrappedLabels entry that pointed at the wrong
#                        source). InitialLabels stems are still refused.
#   include_train_pool - also fold BootstrappedLabels + CorrectedTiles +
#                        mini-crops into the throwaway. Slower but the
#                        starting PRED is much sharper (the pool has way
#                        more polymer signal than 5 InitialLabels alone).
#                        Recommended once the pool has accumulated good labels.
#
# Format note: the PRED is written **1-based** so ilastik can render and
# paint it. The training reader (load_bootstrap_label) auto-detects the
# convention and shifts on the fly — no manual conversion needed before §4.
from sickling.protrusion_detection.bootstrap import run_bootstrap

rep = run_bootstrap(
    throwaway_epochs=15,
    do_final_train=False,
    force_stems=['D16_03_1_14_Bright Field_001'],
    include_train_pool=True,
)
print(rep)

## §3. Correction round (main HITL loop)

Picks `cfg.PRED_BATCH_SIZE` pool images (missing-PRED-first, then oldest-mtime), refreshes their PREDs with the current best model, mines `cfg.MINING_BATCH_SIZE` FN-aware tiles, and stages them in `TilesToBeCorrected` for painting in ilastik.

After this runs, hop to ilastik:
- Open each tile's raw `.jpg`, import the matching `PRED_..._labels.h5` as Labels.
- Paint **only** pixels you're sure about. Leave the rest untouched — it stays unannotated and is ignored in training.
- Save back to `CorrectedTiles/` keeping the exact filename (`<stem>__y<top>_x<left>_labels.h5`).

In [3]:
# §3.0 — (OPTIONAL, ON DEMAND) Full pool PRED refresh.
#
# run_correction_round below only rewrites PREDs for the first
# cfg.PRED_BATCH_SIZE pool images each call. After a retrain the rest of the
# pool's PREDs are stale. Run THIS cell once when you want every eligible
# pool image's PRED regenerated against the current best model (e.g. before
# mining mini-crops across the whole pool). One forward pass per image —
# skip it on rounds where it's not worth the wall-clock.
from sickling.protrusion_detection.correction import regenerate_pool_preds
from sickling.protrusion_detection.splits import build_val_test_pairs

val_pairs, _ = build_val_test_pairs()
n = regenerate_pool_preds(val_pairs=val_pairs)
print(f'wrote {n} fresh PREDs across the pool.')

🔄 Full pool PRED refresh: 341 images  | model: unet_fold_1_best_loop_5.pth


E:\utku g leica\sickling\sickling\protrusion_detection\py_modules\inference.py:152: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt_path,

Pool PRED refresh:   0%|          | 0/341 [00:00<?, ?it/s]

wrote 341 fresh PREDs across the pool.


In [ ]:
from sickling.protrusion_detection.correction import run_correction_round
from sickling.protrusion_detection.splits import build_val_test_pairs

val_pairs, _ = build_val_test_pairs()
report = run_correction_round(val_pairs=val_pairs)
print(f'predicted on {report.n_predicted} pool images, scored {report.n_candidates} candidates, staged {report.n_staged} tiles.')
print(f'best ckpt used: {report.best_ckpt}')
for p in report.picks[:5]:
    print(f'  staged: {p.stem[:40]}__y{p.top}_x{p.left}  score={p.score:.4f}  soft={p.soft_prob:.3f}  dis={p.disagreement:.3f}')

In [ ]:
# §3.1 — (ALTERNATIVE) Mini-crops mining: many tiny adaptive crops, one class.
#
# Use this when dense 512-px tiles have plateaued and your labelling time is
# the bottleneck. Mines hundreds of small crops (bbox + cfg.MINI_CROP_PADDING,
# clamped to [cfg.MINI_CROP_MIN, cfg.MINI_CROP_MAX]) centered on predicted
# cfg.MINI_CROP_TARGET_CLASS connected components, ranked by soft-prob
# (+ disagreement in kfold). You paint **only the target class** in each
# crop; the densify-on-retrain step (§4) auto-fills non-target pixels from
# the clean (loop_0) model so this never reintroduces the positive-only-tile
# over-firing.
#
# Skip semantics: open a crop in ilastik, decide you can't tell, close it
# without saving — that's a skip. Next time this cell runs it sweeps every
# unpainted crop staged before the latest retrain to MiniTilesToBeCorrected/
# _skipped/ and stages new ones in their place.
#
# Output: MiniTilesToBeCorrected/<stem>__y..._x..._h..._w....jpg  (raw)
#         MiniTilesToBeCorrected/PRED_<stem>__y..._x..._h..._w....h5  (PRED)
# Paint and save:
#         MiniTilesCorrected/<stem>__y..._x..._h..._w..._labels.h5
from sickling.protrusion_detection.minicrops import mine_mini_crops
from sickling.protrusion_detection.splits import build_val_test_pairs

val_pairs, _ = build_val_test_pairs()
mreport = mine_mini_crops(val_pairs=val_pairs)
print(f'eligible pool: {mreport.n_eligible}  inferred: {mreport.n_inferred}  '
      f'candidates: {mreport.n_candidates}  staged: {mreport.n_staged}')
print(f'best ckpt used: {mreport.best_ckpt}')
for p in mreport.picks[:5]:
    print(f'  staged: {p.stem[:40]}__y{p.top}_x{p.left}_h{p.h}_w{p.w}  '
          f'score={p.score:.3f}  soft={p.soft_prob:.3f}  dis={p.disagreement:.3f}  '
          f'cc_area={p.cc_area}')

## §4. Retrain (after painting)

In [ ]:
from sickling.protrusion_detection.train import run_training
report = run_training()  # folds=None defaults to BEST_FOLD in single mode, all folds in kfold mode
for f, ckpt, best, last in report.folds:
    cls = ', '.join(f'c{c}={last[c]:.3f}' for c in range(len(last)))
    print(f'fold {f}: best target dice = {best:.3f}  ({cls})  -> {ckpt}')

## §5. Trajectory + final evaluation

In [4]:
# §5.0 — (ON DEMAND) Rebuild visual overlays under viz/.
#
# Writes raw-plus-class-overlay PNGs that mirror the source folder layout:
#   viz/InitialLabels/<stem>.png            — dense val/test masks
#   viz/BootstrappedLabels/<stem>.png       — dense anchor masks
#   viz/CorrectedTiles/<stem>__y..._x...png — painted/densified tiles
#   viz/TilesToBeCorrected/PRED_<...>.png   — staged tiles' model PREDs
#   viz/CorrectionPool/PRED_<stem>.png      — current model PRED on pool
#
# All overlays share the same color map (polymer = red, body = dark teal,
# boundary = green, bg/ignore = raw shows through) so the same color always
# means the same class — easy to compare across folders. Useful for:
#   - sanity-checking InitialLabels GT before chasing val_dice further,
#   - eyeballing where the model finds polymer on pool images,
#   - triaging staged tiles before painting (skip the obviously-wrong ones).
#
# pool_pred_max_n caps the pool sweep (it's the slow one). Set to None to
# render every PRED in CorrectionPool.
from sickling.protrusion_detection.viz import render_all_overlays

counts = render_all_overlays(pool_pred_max_n=100)
for k, v in counts.items():
    print(f'{k:20s} -> {v} PNGs')
print(f'Open: {os.path.join(cfg.VIZ_DIR)}')

InitialLabels        -> 5 PNGs
BootstrappedLabels   -> 12 PNGs
CorrectedTiles       -> 90 PNGs
TilesToBeCorrected   -> 30 PNGs
CorrectionPool       -> 100 PNGs
Open: E:\utku g leica\sickling\sickling\protrusion_detection\viz


In [ ]:
from sickling.protrusion_detection.metrics import plot_trajectory, plot_tp_fp_trajectory, trajectory_csv_path
csv_path = trajectory_csv_path()
png_path = plot_trajectory()
tpfp_path = plot_tp_fp_trajectory()  # TP rate + FP fraction for cfg.TARGET_CLASSES
print(f'trajectory CSV : {csv_path}')
print(f'trajectory PNG : {png_path}')
print(f'TP/FP PNG      : {tpfp_path}')

In [ ]:
# Panel B/C evaluation on the held-out InitialLabels test split.
# (Add an eval helper here when you want poster figures; the bones live in
# sickling.protrusion_detection.metrics for per-class dice + confusion matrix.)

## §6. Sealed held-out eval set (mini-crop test surface)

Hand-labelled mini-crops as a **held-out test set** for the polymer-class recall/precision question. The 2 InitialLabels test surface is too small and too polymer-rare to give a reliable recall estimate; aggregating polymer TP/FP/FN across ~100 hand-labelled mini-crops does.

**Sealed leakage** — both directions:
- Staging skips any source stem in InitialLabels, BootstrappedLabels, or with any tile in CorrectedTiles / MiniTilesCorrected.
- `build_train_pool` automatically excludes every source stem present in `MiniTilesForEval/`.
So no source image can simultaneously contribute training pixels and eval pixels.

**Workflow:**
1. Run §6.1 to stage `cfg.MINI_EVAL_N_CROPS` raw 192-px crops into `MiniTilesForEval/` (no PRED).
2. Open each in ilastik, **dense-label all four classes from scratch**, export as `<crop-base>_labels.h5` in the same folder.
3. Re-run §4. The retrain auto-evaluates against the eval set and appends `eval_recall_class_*` / `eval_precision_class_*` columns to `metrics/iteration_log.csv`. From this loop forward, that's your robust polymer TP signal.

In [ ]:
# §6.1 — Stage sealed, polymer-centered eval crops with 2x context + frame.
#
# Each crop:
#   * Saved image is 2x the eval region on each side (so you see polymer
#     plus surrounding cells/context — no more cells cut off at the edge).
#   * The eval region is the **center** of the image, marked with a 1-px
#     white frame just outside its boundary.
#   * Paint accurately INSIDE the white frame. Outside the frame is
#     visual context only — never graded for TP/FP/FN, so paint it
#     roughly or skip it entirely.
#   * Eval region size is adaptive: bbox + cfg.MINI_EVAL_CROP_PADDING,
#     clamped to [cfg.MINI_EVAL_CROP_MIN, cfg.MINI_EVAL_CROP_MAX]. Saved
#     image is twice that on each side.
#
# Idempotent: rerunning only adds new stems (existing crops in
# MiniTilesForEval/ keep their stems sealed).
#
# Knobs:
#   n_crops           - default cfg.MINI_EVAL_N_CROPS = 100.
#   balance_by_well   - round-robin across wells (first 3 chars of stem).
#                       Without this, dominant well = ~half the crops.
#
# IMPORTANT: if you have leftover crops from v1 (fixed-size random) or v2
# (no-context CC-centered), DELETE MiniTilesForEval/ contents first.
# The evaluator auto-detects the v2 vs v3 layout, so mixing works in
# theory, but the per-well metric is cleaner if every crop has matching
# context.
from sickling.protrusion_detection.minicrops import stage_eval_minicrops

written = stage_eval_minicrops(
    n_crops=cfg.MINI_EVAL_N_CROPS,
    seed=0,                     # change to re-roll the random pick
    balance_by_well=True,
)
print(f'\n{len(written)} polymer-centered eval crops in {cfg.MINI_TILES_FOR_EVAL_DIR}.')
print('Open each in ilastik. Paint ALL FOUR CLASSES carefully INSIDE the '
      'white frame; the area outside the frame is context only. Save as '
      "<base>_labels.h5 in the same folder.")